# ForgeLM — Multi-Dataset Training

Mix multiple datasets with configurable ratios in a single training run.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/multi_dataset.ipynb)

In [ ]:
!pip install -q git+https://github.com/cemililik/ForgeLM.git

In [ ]:
import json

# Dataset 1: General Q&A
general = [
    {
        "messages": [
            {"role": "user", "content": "What is AI?"},
            {"role": "assistant", "content": "AI is artificial intelligence."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is ML?"},
            {"role": "assistant", "content": "ML is machine learning."},
        ]
    },
]

# Dataset 2: Domain-specific (coding)
coding = [
    {
        "messages": [
            {"role": "user", "content": "Write hello world in Python"},
            {"role": "assistant", "content": "print('Hello, World!')"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a list?"},
            {"role": "assistant", "content": "A list is a mutable sequence in Python."},
        ]
    },
]

for name, data in [("general.jsonl", general), ("coding.jsonl", coding)]:
    with open(name, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

print("Datasets created!")

In [ ]:
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-1.7B-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  output_dir: "./multi_checkpoints"
  num_train_epochs: 2
  per_device_train_batch_size: 2
  learning_rate: 2.0e-5

data:
  dataset_name_or_path: "general.jsonl"   # Primary dataset
  extra_datasets:                          # Additional datasets
    - "coding.jsonl"
  mix_ratio: [0.7, 0.3]                   # 70% general, 30% coding
"""

with open("multi_config.yaml", "w") as f:
    f.write(config_yaml)
print("Config saved!")

In [ ]:
!forgelm --config multi_config.yaml --dry-run

In [ ]:
!forgelm --config multi_config.yaml